# Statistical Arbitrage: Pairs Trading Strategy Analysis

## Executive Overview
Statistical arbitrage pairs trading exploits temporary divergences in the equilibrium relationship between two cointegrated assets. Unlike pure correlation (which measures co-movement of returns but does not prevent random walk drift), **cointegration** guarantees that a stationary linear combination exists:

$$Y_t - \beta X_t - \alpha = \epsilon_t, \quad \epsilon_t \sim I(0)$$

This notebook demonstrates:
1. **Correlation screening and Engle-Granger two-step cointegration testing**
2. **Ornstein-Uhlenbeck process modelling** to estimate spread mean-reversion half-life
3. **Dynamic state-space estimation** using a **Kalman Filter** vs static Ordinary Least Squares (OLS)
4. **Z-score signal generation** with mean-reversion entry, profit-taking, and risk-management stop-loss thresholds
5. **Backtesting with institutional market friction** (broker commission, slippage, and short borrow rates)
6. **Multi-pair portfolio diversification** and **walk-forward out-of-sample validation**

In [ ]:
# Imports and environment configuration
import os
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data.data_loader import fetch_historical_prices, DEFAULT_UNIVERSE
from src.pair_selection import screen_correlation, test_cointegration, find_cointegrated_pairs
from src.spread import calculate_ols_spread, calculate_kalman_spread, calculate_rolling_zscore
from src.signals import generate_trading_signals
from src.backtest import run_backtest
from src.metrics import calculate_performance_metrics
from src.portfolio import run_multipair_portfolio, run_walk_forward_validation

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

## 1. Candidate Universe Ingestion
We pull historical closing prices across sector-matched equities (Energy and Financials).

In [ ]:
prices = fetch_historical_prices(tickers=DEFAULT_UNIVERSE, cache_path='../data/prices.csv')
print(f'Ingested {len(prices)} trading days for {len(prices.columns)} assets.')
prices.head()

## 2. Correlation Screening & Engle-Granger Cointegration Testing
We test all asset pairs with Pearson correlation $r \ge 0.70$ for Engle-Granger cointegration ($p < 0.05$) and verify the stationarity of residuals using the Augmented Dickey-Fuller (ADF) test.

In [ ]:
coint_results = find_cointegrated_pairs(prices, min_corr=0.70, max_pvalue=0.05, max_half_life=60.0)
if coint_results.empty:
    coint_results = find_cointegrated_pairs(prices, min_corr=0.60, max_pvalue=0.10, max_half_life=90.0)

print('Identified Cointegrated Pairs:')
coint_results

## 3. Dynamic Kalman Filter vs Static OLS Hedge Ratio
A static OLS regression fits a single slope parameter $\beta$ over the entire historical window, assuming fixed long-term structural parameters. However, company fundamentals, capital structures, and market regimes shift over time.

We formulate a **Kalman Filter state-space model** to estimate a time-varying hedge ratio $\beta_t$ dynamically without lookahead bias:

$$\theta_t = \theta_{t-1} + w_t, \quad w_t \sim \mathcal{N}(0, Q)$$
$$Y_t = [1, X_t] \theta_t + v_t, \quad v_t \sim \mathcal{N}(0, R)$$

In [ ]:
best_pair = coint_results.iloc[0]
ticker_y = best_pair['Asset_Y']
ticker_x = best_pair['Asset_X']
print(f'Selected Best Pair: {ticker_y} on {ticker_x}')

py = prices[ticker_y]
px = prices[ticker_x]

# Static OLS
ols_beta, ols_alpha, ols_spread = calculate_ols_spread(py, px)

# Dynamic Kalman Filter
kf_res = calculate_kalman_spread(py, px, delta=1e-4, R=1e-3)
kf_beta = kf_res['beta']
kf_spread = kf_res['spread']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.plot(prices.index, ols_spread, label=f'OLS Spread (Beta={ols_beta:.3f})', linestyle='--', color='gray')
ax1.plot(prices.index, kf_spread, label='Kalman Filter Spread', color='green')
ax1.axhline(0, color='black', linestyle=':')
ax1.set_title(f'Spread Comparison: {ticker_y} vs {ticker_x}', fontweight='bold')
ax1.legend()

ax2.plot(prices.index, kf_beta, label='Dynamic Kalman Beta', color='crimson')
ax2.axhline(ols_beta, label=f'Static OLS Beta ({ols_beta:.3f})', linestyle='--', color='navy')
ax2.set_title('Hedge Ratio Evolution', fontweight='bold')
ax2.legend()
plt.tight_layout()
plt.show()

## 4. Z-Score Construction & Trading Signals
We standardise the dynamic spread using a rolling 30-day window to calculate $Z_t = \frac{S_t - \mu_t}{\sigma_t}$.

- **Enter Long Spread**: $Z_t \le -2.0\sigma$
- **Enter Short Spread**: $Z_t \ge +2.0\sigma$
- **Mean Reversion Exit**: $Z_t$ reaches $0.0\sigma$
- **Stop-Loss Liquidation**: $|Z_t| \ge 4.0\sigma$

In [ ]:
z_df = calculate_rolling_zscore(kf_spread, window=30)
signals_df = generate_trading_signals(
    z_df['zscore'],
    entry_threshold=2.0,
    exit_threshold=0.0,
    stop_loss_threshold=4.0
)

fig, ax = plt.subplots(figsize=(14, 5))
z = z_df['zscore']
ax.plot(z.index, z, label='Spread Z-Score', color='#1f77b4')
ax.axhline(2.0, color='red', linestyle='--', label='Short Entry (+2σ)')
ax.axhline(-2.0, color='green', linestyle='--', label='Long Entry (-2σ)')
ax.axhline(0.0, color='gray', linestyle=':', label='Mean Exit (0σ)')
ax.axhline(4.0, color='purple', linestyle='-.', label='Stop Loss (+4σ)')
ax.axhline(-4.0, color='purple', linestyle='-.', label='Stop Loss (-4σ)')

longs = z.index[signals_df['signal'] == 1]
shorts = z.index[signals_df['signal'] == -1]
exits = z.index[signals_df['signal'] == 2]
stops = z.index[signals_df['signal'] == -2]

if len(longs) > 0:
    ax.scatter(longs, z.loc[longs], marker='^', color='green', s=60, label='Long Entry')
if len(shorts) > 0:
    ax.scatter(shorts, z.loc[shorts], marker='v', color='red', s=60, label='Short Entry')
if len(exits) > 0:
    ax.scatter(exits, z.loc[exits], marker='o', color='blue', s=45, label='Exit')
if len(stops) > 0:
    ax.scatter(stops, z.loc[stops], marker='x', color='purple', s=60, label='Stop Loss')

ax.set_title(f'Z-Score & Executed Trades ({ticker_y}-{ticker_x})', fontweight='bold')
ax.legend(ncol=3)
plt.tight_layout()
plt.show()

## 5. Backtesting with Institutional Friction & Performance Evaluation
We test the strategy with $100,000 capital, 5 bps commission, 5 bps slippage, and 1.0% annualised short borrow rate. Execution is lagged by 1 period to strictly eliminate lookahead bias.

In [ ]:
backtest_res = run_backtest(
    price_y=py,
    price_x=px,
    hedge_ratio=kf_beta,
    positions=signals_df['position'],
    initial_capital=100000.0,
    commission_bps=5.0,
    slippage_bps=5.0,
    borrow_rate_bps=100.0
)

daily_df = backtest_res['daily_df']
trades_df = backtest_res['trades']

metrics = calculate_performance_metrics(
    daily_returns=daily_df['daily_return'],
    equity_series=daily_df['equity'],
    trades_df=trades_df
)

pd.DataFrame(list(metrics.items()), columns=['Metric', 'Value'])

## 6. Equity Curve & Drawdown Analysis

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
ax1.plot(daily_df.index, daily_df['equity'], color='forestgreen', linewidth=1.8, label='Pairs Trading Strategy')
ax1.axhline(100000, color='black', linestyle=':')
ax1.set_title('Portfolio Equity Curve ($)', fontweight='bold')
ax1.legend()

peak = daily_df['equity'].cummax()
dd = ((daily_df['equity'] - peak) / peak) * 100.0
ax2.fill_between(daily_df.index, dd, 0, color='crimson', alpha=0.3)
ax2.plot(daily_df.index, dd, color='crimson', linewidth=1.2)
ax2.set_title('Underwater Drawdown (%)', fontweight='bold')
ax2.set_ylabel('Drawdown (%)')
plt.tight_layout()
plt.show()

## 7. Multi-Pair Portfolio & Walk-Forward Out-of-Sample Validation
We now evaluate portfolio-level risk diversification across the top 3 cointegrated pairs, followed by a walk-forward out-of-sample validation to examine performance stability over rolling time windows.

In [ ]:
port_res = run_multipair_portfolio(prices, coint_results, max_pairs=3, initial_capital=100000.0)
print('Multi-Pair Portfolio Performance Metrics:')
pd.DataFrame(list(port_res['portfolio_metrics'].items()), columns=['Metric', 'Value'])

In [ ]:
wf_res = run_walk_forward_validation(prices, in_sample_len=252, out_of_sample_len=63)
print('Walk-Forward Window Summaries:')
wf_res['windows']